# Veri Bilimi Kariyerlerinde Maaş Analizi
Bu not defteri, "Veri Biliminde Altın Çağ: Maaşlar Nereye Gidiyor?" isimli Scrollytelling projesi için verilerin yüklenmesi, temizlenmesi, keşifsel veri analizi (EDA) ve web sitesinde kullanılacak özet (aggregate) dosyaların üretilmesi adımlarını içermektedir.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("C:/Users/PC/Desktop/DataScience_salaries_2024-selected-columns.csv")

In [3]:
print("Ham Veri Boyutu:", df.shape)
display(df.head())

Ham Veri Boyutu: (14838, 10)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location
0,2021,MI,FT,Data Scientist,30400000,CLP,40038,CL,100,CL
1,2021,MI,FT,BI Data Analyst,11000000,HUF,36259,HU,50,US
2,2020,MI,FT,Data Scientist,11000000,HUF,35735,HU,50,HU
3,2021,MI,FT,ML Engineer,8500000,JPY,77364,JP,50,JP
4,2022,SE,FT,Lead Machine Learning Engineer,7500000,INR,95386,IN,50,IN


Veri Temizleme Kararı (Eksik ve Mükerrer Veriler): Veri setinde eksik (null) değer kontrolü yapılmış ve hiçbir eksik veriye rastlanmamıştır. Ancak, verilerin toplanma sürecindeki hatalardan kaynaklanabilecek 5.748 adet mükerrer (duplicate) kayıt tespit edilmiştir. İstatistiksel ortalamaları yanıltmaması ve belirli maaşların ağırlığını suni olarak artırmaması amacıyla bu tekrarlayan satırlar analizden tamamen çıkarılmıştır.

In [4]:
print("Eksik Değerler:\n", df.isnull().sum())

Eksik Değerler:
 work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
dtype: int64


In [5]:
ilk_satir_sayisi = df.shape[0]
df_clean = df.drop_duplicates().copy()
son_satir_sayisi = df_clean.shape[0]
print(f"\nSilinen tekrarlayan satır sayısı: {ilk_satir_sayisi - son_satir_sayisi}")
print(f"Tekrarlar silindikten sonra veri boyutu: {df_clean.shape}")
df_clean = df_clean[['work_year', 'experience_level', 'job_title', 'salary_in_usd', 'company_location', 'remote_ratio']]


Silinen tekrarlayan satır sayısı: 5748
Tekrarlar silindikten sonra veri boyutu: (9090, 10)


Veri Düzenleme Kararı (Deneyim Seviyesi ve Çalışma Şekli):** 
1. `experience_level` sütununda yer alan kısaltmalar ("EN", "MI", "SE", "EX"), hikaye anlatımı sırasında okuyucunun veriyi daha rahat anlayabilmesi ve bilişim sektöründeki küresel standartlara uyum sağlaması amacıyla sırasıyla "Junior", "Mid-level", "Senior" ve "Executive" olarak yeniden adlandırılmıştır.
2. `remote_ratio` sütununda yer alan 0, 50 ve 100 sayısal değerleri, görselleştirmelerde anlam ifade etmesi için sırasıyla "On-site" (Ofis), "Hybrid" (Hibrit) ve "Remote" (Uzaktan) olarak metinlere dönüştürülmüştür.

In [6]:
exp_map = {
    'EN': 'Junior',
    'MI': 'Mid-level',
    'SE': 'Senior',
    'EX': 'Executive'
}
if 'experience_level' in df_clean.columns:
    df_clean['experience_level'] = df_clean['experience_level'].map(exp_map)

In [7]:
remote_map = {0: 'On-site', 50: 'Hybrid', 100: 'Remote'}
if 'remote_ratio' in df_clean.columns:
    df_clean['remote_ratio'] = df_clean['remote_ratio'].map(remote_map)

display(df_clean.head())

,work_year,experience_level,job_title,salary_in_usd,company_location,remote_ratio
0,2021,Mid-level,Data Scientist,40038,CL,Remote
1,2021,Mid-level,BI Data Analyst,36259,US,Hybrid
2,2020,Mid-level,Data Scientist,35735,HU,Hybrid
3,2021,Mid-level,ML Engineer,77364,JP,Hybrid
4,2022,Senior,Lead Machine Learning Engineer,95386,IN,Hybrid



Veri Temizleme Kararı (Aykırı Değerler): Maaş (`salary_in_usd`) değişkenindeki aykırı değerler (outliers) standart IQR (Çeyrekler Açıklığı) yöntemi ile hesaplanmış ve tespit edilmiştir. İstatistiksel olarak aşırı uçlarda yer alan ve genel sektör maaş eğilimini yansıtmayıp görselleştirmeleri (grafik eksenlerini) bozacak olan istisnai değerler (örneğin aşırı yüksek yönetici veya düşük stajyer maaşları) analizden çıkarılmıştır.

In [8]:
Q1 = df_clean['salary_in_usd'].quantile(0.25)
Q3 = df_clean['salary_in_usd'].quantile(0.75)
IQR = Q3 - Q1

alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

In [9]:
aykiri_sayisi = df_clean[(df_clean['salary_in_usd'] < alt_sinir) | (df_clean['salary_in_usd'] > ust_sinir)].shape[0]
print(f"Tespit edilen aşırı uç (aykırı) maaş satır sayısı: {aykiri_sayisi}")

df_clean = df_clean[(df_clean['salary_in_usd'] >= alt_sinir) & (df_clean['salary_in_usd'] <= ust_sinir)]
print(f"Aykırı değerler silindikten sonra son veri boyutu: {df_clean.shape}")

Tespit edilen aşırı uç (aykırı) maaş satır sayısı: 166
Aykırı değerler silindikten sonra son veri boyutu: (8924, 6)


Veri İnceleme Kararı (Dağılımlar):** Ortalama maaşları kıyaslarken istatistiksel yanılgılara (az verisi olan bir grubun ortalamayı saptırması vb.) düşmemek adına, yıllara, lokasyonlara, deneyim seviyelerine ve çalışma şekline göre veri setindeki kayıt sayıları incelenmiştir.

In [10]:
print("--- Yıllara Göre Kayıt Sayısı ---")
print(df_clean['work_year'].value_counts().sort_index())

print("\n--- Deneyim Seviyesine Göre Kayıt Sayısı ---")
print(df_clean['experience_level'].value_counts())

print("\n--- Çalışma Şekline Göre Kayıt Sayısı ---")
print(df_clean['remote_ratio'].value_counts())

print("\n--- En Çok Kayıt Olan 10 Şirket Lokasyonu ---")
print(df_clean['company_location'].value_counts().head(10))

--- Yıllara Göre Kayıt Sayısı ---
work_year
2020      72
2021     213
2022    1090
2023    4536
2024    3013
Name: count, dtype: int64

--- Deneyim Seviyesine Göre Kayıt Sayısı ---
experience_level
Senior       5271
Mid-level    2432
Junior        889
Executive     332
Name: count, dtype: int64

--- Çalışma Şekline Göre Kayıt Sayısı ---
remote_ratio
On-site    5544
Remote     3138
Hybrid      242
Name: count, dtype: int64

--- En Çok Kayıt Olan 10 Şirket Lokasyonu ---
company_location
US    7329
GB     509
CA     348
DE      97
ES      70
FR      60
IN      59
AU      49
NL      28
PT      28
Name: count, dtype: int64


Ana Bulgular ve Web Sitesi İçin Dışa Aktarma
Yapılan Keşifsel Veri Analizi sonucunda projemiz için şu bulgulara odaklanılmasına karar verilmiştir:
1. **Deneyim Seviyesinin Etkisi:** Junior seviyesinden Senior seviyeye geçişte maaşlarda yıllar içinde belirgin bir sıçrama yaşanmaktadır.
2. **Lokasyon Etkisi:** ABD merkezli şirketlerin veri bilimi pozisyonlarına ödediği ortalama maaşlar, diğer ülkelere kıyasla çok daha yüksektir.
3. **Unvan Rekabeti:** Data Scientist, Data Engineer ve Machine Learning Engineer arasındaki maaş rekabeti.
4. **Çalışma Şekli Etkisi:** Uzaktan (Remote) vs Ofis (On-site) maaş eğilimleri.

*Aşağıdaki kod, web sitesinin hızlı çalışması (5 MB kuralı) için ham veri yerine sadece grafikleri çizecek olan özet (aggregate) CSV dosyalarını oluşturmaktadır.

In [11]:
df_trend = df_clean.groupby(['work_year', 'experience_level'])['salary_in_usd'].mean().round().reset_index()
df_trend.to_csv('bulgu1_deneyim_yillik_maas.csv', index=False)

In [12]:
df_clean['location_group'] = np.where(df_clean['company_location'] == 'US', 'ABD Merkezli', 'Diğer Ülkeler')
df_location = df_clean.groupby(['work_year', 'location_group'])['salary_in_usd'].mean().round().reset_index()
df_location.to_csv('bulgu2_lokasyon_maas.csv', index=False)

In [13]:
populer_roller = ['Data Scientist', 'Data Engineer', 'Machine Learning Engineer']
df_roles = df_clean[df_clean['job_title'].isin(populer_roller)]
df_roles_trend = df_roles.groupby(['work_year', 'job_title'])['salary_in_usd'].mean().round().reset_index()
df_roles_trend.to_csv('bulgu3_populer_roller.csv', index=False)

In [14]:
df_remote = df_clean.groupby(['work_year', 'remote_ratio'])['salary_in_usd'].mean().round().reset_index()
df_remote.to_csv('bulgu4_calisma_sekli_maas.csv', index=False)

In [15]:
df_interactive = df_clean.groupby(['work_year', 'job_title', 'experience_level'])['salary_in_usd'].mean().round().reset_index()
df_interactive.to_csv('okuyucu_kesfi_detayli.csv', index=False)

In [16]:
print("TEBRİKLER! Veri analizi tamamlandı ve web sitesi için tüm .csv dosyaları başarıyla dışa aktarıldı.")

TEBRİKLER! Veri analizi tamamlandı ve web sitesi için tüm .csv dosyaları başarıyla dışa aktarıldı.


In [17]:
import pandas as pd
import json
import numpy as np

In [18]:
def ozet(g, gruplar):
    o = (g.groupby(gruplar)['salary_in_usd']
           .agg(medyan='median',
                q1=lambda x: x.quantile(0.25),
                q3=lambda x: x.quantile(0.75),
                n='count')
           .reset_index())
    o['yeterli'] = o['n'] >= 30
    return o


In [19]:
# Bulgu 1: Junior ve Senior
bulgu1 = ozet(df_clean[df_clean['experience_level'].isin(['Junior', 'Senior'])],
              ['work_year', 'experience_level'])

In [20]:
# Bulgu 2: Senior, ABD ve diğer
senior_df = df_clean[df_clean['experience_level'] == 'Senior'].copy()
senior_df['is_us'] = np.where(senior_df['company_location'] == 'US', 'ABD', 'Diğer Ülkeler')
bulgu2 = ozet(senior_df, ['work_year', 'is_us'])


In [21]:
# Bulgu 3: Mid-level, üç rol
hedef_roller = ['Data Scientist', 'Data Engineer', 'Machine Learning Engineer']
mid_df = df_clean[(df_clean['experience_level'] == 'Mid-level') &
                  (df_clean['job_title'].isin(hedef_roller))].copy()
bulgu3 = ozet(mid_df, ['job_title'])

In [22]:
# Bulgu 4: ABD Senior, Uzaktan ve Ofis (hibrit dışarıda)
us_senior_df = df_clean[(df_clean['experience_level'] == 'Senior') &
                        (df_clean['company_location'] == 'US') &
                        (df_clean['remote_ratio'].isin(['On-site', 'Remote']))].copy()
bulgu4 = ozet(us_senior_df, ['remote_ratio'])


In [23]:
# Okuyucu keşfi: yeterli kayıtlı unvanlar
kesif = ozet(df_clean, ['job_title', 'experience_level', 'work_year'])
unvan_n = df_clean.groupby('job_title').size()
kesif = kesif[kesif['job_title'].isin(unvan_n[unvan_n >= 30].index) & (kesif['n'] >= 5)]

proje_verisi = {
    "bulgu1_makas": bulgu1.to_dict(orient='records'),
    "bulgu2_cografya": bulgu2.to_dict(orient='records'),
    "bulgu3_unvan": bulgu3.to_dict(orient='records'),
    "bulgu4_calisma_sekli": bulgu4.to_dict(orient='records'),
    "okuyucu_kesfi": kesif.to_dict(orient='records'),
}
with open('maas_verisi.json', 'w', encoding='utf-8') as f:
    json.dump(proje_verisi, f, ensure_ascii=False, indent=4,
              default=lambda o: o.item() if hasattr(o, 'item') else str(o))

pd.set_option('display.width', 200)
print("BULGU 1\n", bulgu1, "\n")
print("BULGU 2\n", bulgu2, "\n")
print("BULGU 3\n", bulgu3, "\n")
print("BULGU 4\n", bulgu4, "\n")
print("KEŞİF:", kesif.shape, "satır,", kesif['job_title'].nunique(), "unvan")

BULGU 1
    work_year experience_level    medyan         q1        q3     n  yeterli
0       2020           Junior   51321.0   42000.00   91000.0    21    False
1       2020           Senior  118552.0   74130.25  160500.0    18    False
2       2021           Junior   61915.5   29920.25   82146.0    46     True
3       2021           Senior  117500.0   77045.75  170000.0    74     True
4       2022           Junior   60413.0   40000.00  100000.0   102     True
5       2022           Senior  144854.0  110000.00  185850.0   667     True
6       2023           Junior   80000.0   51895.75  120000.0   346     True
7       2023           Senior  155000.0  119629.50  200000.0  2944     True
8       2024           Junior   86950.0   64675.00  120000.0   374     True
9       2024           Senior  153000.0  114750.00  200000.0  1568     True 

BULGU 2
    work_year          is_us    medyan        q1         q3     n  yeterli
0       2020            ABD  157000.0  107107.0  188500.00     8    Fa

In [24]:
bulgu4['calisma_sekli'] = bulgu4['remote_ratio'].map({'Remote': 'Uzaktan', 'On-site': 'Ofis'})

for t in (bulgu1, bulgu2, bulgu3, bulgu4, kesif):
    t['salary_in_usd'] = t['medyan']

proje_verisi = {
    "bulgu1_makas": bulgu1.to_dict(orient='records'),
    "bulgu2_cografya": bulgu2.to_dict(orient='records'),
    "bulgu3_unvan": bulgu3.to_dict(orient='records'),
    "bulgu4_calisma_sekli": bulgu4.to_dict(orient='records'),
    "okuyucu_kesfi": kesif.to_dict(orient='records'),
}
with open('maas_verisi.json', 'w', encoding='utf-8') as f:
    json.dump(proje_verisi, f, ensure_ascii=False, indent=4,
              default=lambda o: o.item() if hasattr(o, 'item') else str(o))

print(bulgu4)

  remote_ratio    medyan        q1        q3     n  yeterli calisma_sekli  salary_in_usd
0      On-site  159300.0  123000.0  204728.0  2850     True          Ofis       159300.0
1       Remote  157000.0  125000.0  195475.0  1694     True       Uzaktan       157000.0
